# Global Stock Index Analysis Pipeline

**Author:** Matheus Elias  
**Dataset:** Historical daily OHLCV data for multiple global stock indices (1965–2021)  
**Stack:** Python · Pandas · Matplotlib  

---

## Objectives

1. **Ingest & validate** raw CSV data
2. **Clean & transform** data into a reliable analytical structure
3. **Engineer features** — daily return, 20-day & 50-day moving averages, rolling volatility, and drawdown
4. **Visualize** each index with a multi-panel technical chart
5. **Compare** all indices side-by-side on normalized performance

---

## 1. Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec

# ── Plotting defaults ────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#0d1117',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

# ── Color palette for each index ─────────────────────────────────────────────
INDEX_COLORS = {
    'NYA':  '#58a6ff',
    'N100': '#3fb950',
    'IXIC': '#d2a8ff',
    'DJI':  '#ffa657',
    'SPX':  '#ff7b72',
}
DEFAULT_COLOR = '#79c0ff'

print('Setup complete.')

## 2. Ingestion & Validation

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
RAW_PATH = 'indexData.csv'   # adjust path as needed

raw_df = pd.read_csv(RAW_PATH)

print(f'Rows loaded : {len(raw_df):,}')
print(f'Columns     : {list(raw_df.columns)}')
print(f'Indices     : {raw_df["Index"].unique()}')
print()
raw_df.head()

In [ ]:
# ── Basic data quality checks ─────────────────────────────────────────────────
print('=== Null counts per column ===')
print(raw_df.isnull().sum())

print()
print('=== Rows per index ===')
print(raw_df.groupby('Index').size().sort_values(ascending=False))

print()
print('=== Date range per index ===')
date_range = (
    raw_df.assign(Date=pd.to_datetime(raw_df['Date']))
    .groupby('Index')['Date']
    .agg(['min', 'max'])
)
print(date_range)

## 3. Cleaning & Transformation

In [ ]:
def clean_index_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean and type-cast raw OHLCV data.

    Steps
    -----
    1. Parse Date to datetime.
    2. Drop rows where Close is null or zero.
    3. Remove duplicate (Index, Date) pairs, keeping the last entry.
    4. Sort by Index and Date.
    5. Select only the columns needed for downstream analysis.
    """
    KEEP_COLS = ['Index', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume']

    clean = (
        df[KEEP_COLS]
        .assign(Date=lambda d: pd.to_datetime(d['Date']))
        .dropna(subset=['Close'])
        .query('Close > 0')
        .drop_duplicates(subset=['Index', 'Date'], keep='last')
        .sort_values(['Index', 'Date'])
        .reset_index(drop=True)
    )

    return clean


clean_df = clean_index_data(raw_df)

print(f'Rows after cleaning : {len(clean_df):,}  (removed {len(raw_df) - len(clean_df):,})')
clean_df.dtypes

## 4. Feature Engineering

In [ ]:
def engineer_features(df: pd.DataFrame,
                      short_window: int = 20,
                      long_window: int  = 50) -> pd.DataFrame:
    """
    Add technical analysis features, computed per index group.

    New columns
    -----------
    daily_return   : percentage change in Close price day-over-day
    ma_short       : rolling mean of Close over `short_window` days
    ma_long        : rolling mean of Close over `long_window` days
    volatility     : rolling std of daily_return over `short_window` days
    rolling_max    : expanding maximum of Close price
    drawdown       : percentage decline from the rolling peak
    """
    def _per_index(g: pd.DataFrame) -> pd.DataFrame:
        g = g.sort_values('Date').copy()

        g['daily_return'] = g['Close'].pct_change() * 100
        g['ma_short']     = g['Close'].rolling(short_window, min_periods=1).mean()
        g['ma_long']      = g['Close'].rolling(long_window,  min_periods=1).mean()
        g['volatility']   = g['daily_return'].rolling(short_window, min_periods=1).std()

        rolling_max       = g['Close'].cummax()
        g['drawdown']     = (g['Close'] - rolling_max) / rolling_max * 100

        return g

    return df.groupby('Index', group_keys=False).apply(_per_index).reset_index(drop=True)


feat_df = engineer_features(clean_df)

print('Feature engineering complete.')
feat_df[['Index', 'Date', 'Close', 'daily_return', 'ma_short', 'ma_long',
         'volatility', 'drawdown']].tail(10)

## 5. Summary Statistics

In [ ]:
def summary_statistics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute annualised performance metrics per index.

    Metrics
    -------
    trading_days    : total number of trading days in the dataset
    start / end     : date range
    total_return    : cumulative return from first to last close (%)
    ann_return      : CAGR approximation (%)
    ann_volatility  : annualised daily std (%)
    sharpe_ratio    : ann_return / ann_volatility  (risk-free = 0)
    max_drawdown    : worst peak-to-trough decline (%)
    """
    rows = []

    for idx, g in df.groupby('Index'):
        g = g.sort_values('Date')
        n_days        = len(g)
        start         = g['Date'].iloc[0].date()
        end           = g['Date'].iloc[-1].date()
        first_close   = g['Close'].iloc[0]
        last_close    = g['Close'].iloc[-1]

        total_ret     = (last_close / first_close - 1) * 100
        years         = n_days / 252
        ann_ret       = ((last_close / first_close) ** (1 / years) - 1) * 100
        ann_vol       = g['daily_return'].std() * np.sqrt(252)
        sharpe        = ann_ret / ann_vol if ann_vol != 0 else np.nan
        max_dd        = g['drawdown'].min()

        rows.append({
            'Index':          idx,
            'Start':          start,
            'End':            end,
            'Trading Days':   n_days,
            'Total Return %': round(total_ret, 2),
            'Ann. Return %':  round(ann_ret,   2),
            'Ann. Vol %':     round(ann_vol,   2),
            'Sharpe Ratio':   round(sharpe,    3),
            'Max Drawdown %': round(max_dd,    2),
        })

    return pd.DataFrame(rows).set_index('Index').sort_values('Ann. Return %', ascending=False)


summary_df = summary_statistics(feat_df)
summary_df

## 6. Visualisation — Per-Index Technical Chart

In [ ]:
def plot_technical_chart(df: pd.DataFrame, index_name: str) -> None:
    """
    Render a 3-panel technical chart for a single index:
      Panel 1 — Close price with 20-day & 50-day moving averages
      Panel 2 — Rolling 20-day volatility (annualised %)
      Panel 3 — Drawdown from rolling peak (%)
    """
    data  = df[df['Index'] == index_name].sort_values('Date')
    color = INDEX_COLORS.get(index_name, DEFAULT_COLOR)

    fig = plt.figure(figsize=(16, 9), constrained_layout=True)
    fig.suptitle(f'{index_name}  —  Technical Analysis', fontsize=15, color='#e6edf3')

    gs = GridSpec(3, 1, figure=fig, height_ratios=[3, 1, 1], hspace=0.08)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    ax3 = fig.add_subplot(gs[2], sharex=ax1)

    # ── Panel 1: Price + MAs ──────────────────────────────────────────────────
    ax1.plot(data['Date'], data['Close'],    color=color,     lw=1.5,  label='Close')
    ax1.plot(data['Date'], data['ma_short'], color='#ffa657', lw=1.0,  ls='--', label='MA-20')
    ax1.plot(data['Date'], data['ma_long'],  color='#ff7b72', lw=1.0,  ls=':',  label='MA-50')
    ax1.set_ylabel('Price')
    ax1.legend(loc='upper left', fontsize=9, framealpha=0.2)
    ax1.grid(True, alpha=0.3)
    plt.setp(ax1.get_xticklabels(), visible=False)

    # ── Panel 2: Volatility ───────────────────────────────────────────────────
    ann_vol = data['volatility'] * np.sqrt(252)
    ax2.fill_between(data['Date'], ann_vol, alpha=0.4, color='#d2a8ff')
    ax2.plot(data['Date'], ann_vol, color='#d2a8ff', lw=1.0)
    ax2.set_ylabel('Vol (ann. %)')
    ax2.grid(True, alpha=0.3)
    plt.setp(ax2.get_xticklabels(), visible=False)

    # ── Panel 3: Drawdown ─────────────────────────────────────────────────────
    ax3.fill_between(data['Date'], data['drawdown'], alpha=0.5, color='#ff7b72')
    ax3.plot(data['Date'], data['drawdown'], color='#ff7b72', lw=1.0)
    ax3.set_ylabel('Drawdown %')
    ax3.set_xlabel('Date')
    ax3.grid(True, alpha=0.3)

    # ── X-axis date formatting ────────────────────────────────────────────────
    ax3.xaxis.set_major_locator(mdates.YearLocator(5))
    ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax3.xaxis.set_minor_locator(mdates.YearLocator(1))

    plt.savefig(f'{index_name}_technical_chart.png', dpi=150,
                bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'Saved → {index_name}_technical_chart.png')


# ── Plot a chart for every index in the dataset ───────────────────────────────
for idx in sorted(feat_df['Index'].unique()):
    plot_technical_chart(feat_df, idx)

## 7. Visualisation — Normalised Performance Comparison

In [ ]:
def plot_normalised_comparison(df: pd.DataFrame) -> None:
    """
    Overlay all indices on a single chart with prices normalised to 100
    at each index's first trading date, enabling apples-to-apples comparison.
    """
    fig, ax = plt.subplots(figsize=(16, 7))
    fig.suptitle('All Indices — Normalised Performance (base = 100)', fontsize=14,
                 color='#e6edf3')

    for idx, g in df.groupby('Index'):
        g       = g.sort_values('Date')
        norm    = g['Close'] / g['Close'].iloc[0] * 100
        color   = INDEX_COLORS.get(idx, DEFAULT_COLOR)
        ax.plot(g['Date'], norm, label=idx, color=color, lw=1.5)

    ax.axhline(100, color='#8b949e', lw=0.8, ls='--', alpha=0.6)
    ax.set_xlabel('Date')
    ax.set_ylabel('Normalised Value (start = 100)')
    ax.legend(fontsize=10, framealpha=0.2, loc='upper left')
    ax.grid(True, alpha=0.3)

    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    plt.savefig('all_indices_normalised.png', dpi=150,
                bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print('Saved → all_indices_normalised.png')


plot_normalised_comparison(feat_df)

## 8. Key Findings

*(Fill in after running the notebook with your actual dataset)*

| Metric | Finding |
|---|---|
| Best annualised return | *(index name)* — *X*% |
| Highest volatility | *(index name)* — *X*% |
| Worst max drawdown | *(index name)* — *–X*% |
| Best Sharpe ratio | *(index name)* — *X* |

---

## 9. Next Steps

- Load data from an API (e.g., Yahoo Finance via `yfinance`) to keep it current
- Add correlation analysis between indices
- Build an interactive version with Plotly or Streamlit
- Extend to include macroeconomic overlays (inflation, interest rates)